## 1. Setup

In [16]:
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/tokenizerv3.pkl .
# !mkdir ./base-models
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/mla-sin.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/gqa-rope-swiglu-moe.pt ./base-models/
# !cp /kaggle/input/datasets/rahulsingh147/models-tokenizer/swiglu-moe-rope-conv.pt ./base-models/

In [17]:
# Imports
import os, gc, copy, time, math, random, re, json, pickle, zipfile, hashlib
from dataclasses import dataclass, replace
from collections import defaultdict, Counter

import numpy as np, pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score

import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

# Hugging Face login Secret
login(token=UserSecretsClient().get_secret("HF"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data location / field names (check these against the EDA cell output) 
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_DIR      = "Phase_4_CJPE"
ID_KEY, TEXT_KEY, LABEL_KEY = "id", "text", "label"
NUM_LABELS = 2

# True -> tiny subsets + separate output dir, to verify the whole pipeline in minutes
SMOKE_TEST = True
OUT = "./cs781_phase4_smoke" if SMOKE_TEST else "./cs781_phase4"
os.makedirs(OUT, exist_ok=True)

# Phase 1 model hyper-parameters
VOCAB_SIZE       = 20_000
TRAIN_BLOCK_SIZE = 2048
N_EMBD           = 768
N_LAYER          = 12
N_HEAD           = 12
N_KV_HEAD        = 4
KV_LATENT_DIM    = N_EMBD // 4

PHASE1_CONFIGS = {"gqa-rope":             {"n_layers": 8,  "d_ff": 4 * 768, "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": False, "use_conv": False},
                  "mla-sin":              {"n_layers": 12, "d_ff": 4 * 768, "attention_type": "mla", "pos_emb_type": "sinusoidal", "use_moe": False, "use_conv": False},
                  "gqa-rope-swiglu-moe":  {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": False},
                  "swiglu-moe-rope-conv": {"n_layers": 12, "d_ff": 1408,    "attention_type": "gqa", "pos_emb_type": "rope",       "use_moe": True,  "use_conv": True, "local_kernel_size": 25},
                  }
MAIN_MODEL = "gqa-rope-swiglu-moe"   # model used for the technique / ablation experiments

# ── Long-document representation ──
CHUNK_LEN      = 512                         # tokens per chunk, including the <s> token
CHUNK_OVERLAP  = 64                          # tokens shared by consecutive chunks
CONTENT_LEN    = CHUNK_LEN - 1               # real tokens per chunk
CHUNK_STRIDE   = CONTENT_LEN - CHUNK_OVERLAP
MAX_CHUNKS     = 8                           # chunks per case fed to the model (memory/speed knob)
CHUNK_STRATEGY = "head_tail"                 # head | tail | head_tail | uniform

# ── Training ──
BATCH_DOCS      = 2          # cases per step
GRAD_ACCUM      = 8          # effective batch = BATCH_DOCS * GRAD_ACCUM
EVAL_BATCH_DOCS = 8
GRAD_CKPT       = False       # if True, recompute activations in backward -> much lower memory
WARMUP_FRAC     = 0.06
WEIGHT_DECAY    = 0.01
HEAD_DROPOUT    = 0.1
FOCAL_GAMMA     = 2.0
ADALORA_ORTH_COEF = 0.01
USE_TUNED_THRESHOLD = True   # submit with the dev-tuned decision threshold (else 0.5)

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 2. Tokenizer

In [18]:
class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.reserved = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved); self.stoi = {}; self.merges = []; self.ranks = {}; self.merged_map = {}; self._cache = {}
        # Regex pattern matching words, numbers, individual punctuation, and whitespace sequences
        self.pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        self.pat = re.compile(self.pattern, re.UNICODE)

    def fit(self, texts, max_unique_words=200000):
        """Train a byte-level BPE tokenizer using inverted indexing for fast merging."""
        raw_counts = Counter()
        iterator = [texts] if isinstance(texts, str) else texts

        # Preserve whitespace tokens when populating initial raw_counts
        for text in tqdm(iterator, desc="Building word vocabulary"):
            for chunk in re.findall(self.pattern, str(text), flags=re.UNICODE):
                raw_counts[chunk] += 1

        if max_unique_words and len(raw_counts) > max_unique_words:
            most_common = raw_counts.most_common(max_unique_words)
            raw_counts = dict(most_common)

        word_freqs = {}; pair_counts = defaultdict(int); pair_to_words = defaultdict(set)
        for word, count in raw_counts.items():
            encoded = word.encode("utf-8")
            byte_word = tuple(encoded[i:i+1] for i in range(len(encoded)))
            word_freqs[byte_word] = count

            for i in range(len(byte_word) - 1):
                pair = (byte_word[i], byte_word[i + 1])
                pair_counts[pair] += count
                pair_to_words[pair].add(byte_word)

        base_bytes = sorted({token for word in word_freqs for token in word})
        self.itos = list(self.reserved) + base_bytes; self.merges = []
        target_merges = self.vocab_size - len(self.itos)

        with tqdm(total=target_merges, desc="Training BPE Merges") as pbar:
            while len(self.itos) < self.vocab_size:
                if not pair_counts:
                    break
                best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p[0], p[1]))
                if pair_counts[best_pair] <= 0:
                    break
                merged_token = best_pair[0] + best_pair[1]
                self.merges.append((best_pair, merged_token))
                self.itos.append(merged_token)
                
                words_to_update = list(pair_to_words[best_pair])
                del pair_counts[best_pair]
                del pair_to_words[best_pair]

                p0, p1 = best_pair
                for word in words_to_update:
                    count = word_freqs.get(word, 0)
                    if count == 0:
                        continue

                    new_word = []; i = 0; n = len(word)
                    while i < n:
                        if i + 1 < n and word[i] == p0 and word[i + 1] == p1:
                            new_word.append(merged_token)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1
                    new_word = tuple(new_word)
                    
                    for j in range(len(word) - 1):
                        old_p = (word[j], word[j + 1])
                        if old_p in pair_counts:
                            pair_counts[old_p] -= count
                            if pair_counts[old_p] <= 0:
                                del pair_counts[old_p]

                    for j in range(len(new_word) - 1):
                        new_p = (new_word[j], new_word[j + 1])
                        pair_counts[new_p] += count
                        pair_to_words[new_p].add(new_word)

                    del word_freqs[word]
                    word_freqs[new_word] = count

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self.merged_map = {pair: merged for pair, merged in self.merges}
        self._cache = {}
        return self

    def encode_word(self, word):
        """Encode single word using memoization and local pair updates."""
        if word in self._cache:
            return self._cache[word]

        encoded = word.encode("utf-8")
        tokens = [encoded[i:i+1] for i in range(len(encoded))]

        if len(tokens) <= 1:
            res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
            self._cache[word] = res
            return res

        while len(tokens) > 1:
            min_rank = float("inf")
            min_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.ranks.get(pair, float("inf"))
                if rank < min_rank:
                    min_rank = rank
                    min_pair = pair

            if min_pair is None or min_rank == float("inf"):
                break

            merged = self.merged_map[min_pair]
            new_tokens = []
            i = 0
            n = len(tokens)
            p0, p1 = min_pair

            while i < n:
                if i + 1 < n and tokens[i] == p0 and tokens[i + 1] == p1:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        self._cache[word] = res
        return res

    def encode(self, text):
        """Encode a sentence while preserving original spaces and newlines."""
        ids = [self.stoi["<s>"]]
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        for chunk in re.findall(pattern, str(text), flags=re.UNICODE):
            ids.extend(self.encode_word(chunk))
        ids.append(self.stoi["</s>"])
        return ids
    
    def decode(self, ids):
        """Decode BPE token IDs back into text seamlessly."""
        byte_list = []
        for idx in ids:
            tok = self.itos[idx] if 0 <= idx < len(self.itos) else "<unk>"
            if tok not in self.reserved and isinstance(tok, bytes):
                byte_list.append(tok)
        try:
            return b"".join(byte_list).decode("utf-8", errors="ignore")
        except Exception:
            return ""

    def get_vocab_size(self):
        return self.vocab_size

def train_tokenizer(train_text: str, vocab_size: int):
    """Train A-1 BPE tokenizer on `train_text` and return the trained tokenizer."""
    tokenizer = ByteLevelBPE(vocab_size)
    tokenizer.fit(train_text)
    return tokenizer
    
# Load Pretrained Tokenizer
with open("tokenizerv3.pkl", "rb") as f:
    tokenizer = pickle.load(f)
BOS, PAD, UNK, EOS = (tokenizer.stoi[t] for t in ("<s>", "<pad>", "<unk>", "</s>"))
assert len(tokenizer.itos) == VOCAB_SIZE
print("tokenizer vocab:", len(tokenizer.itos))

tokenizer vocab: 20000


## 3. Model Class Definitions

In [19]:
@dataclass
class LMConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = N_EMBD
    n_layers: int = N_LAYER
    n_heads: int = N_HEAD
    n_kv_heads: int = N_KV_HEAD
    kv_latent_dim: int = KV_LATENT_DIM
    d_ff: int = 1408          # For SwiGLU (2/3) * 4 * d_model => 2/3 * 2048 = 1365 --> round to nearest upper multiple of 64 (1408)
    max_seq_len: int  = TRAIN_BLOCK_SIZE
    dropout: float = 0.0            # ~1 pass over the data -> no need for dropout
    pos_emb_type: str = "rope"      # "rope" or "sinusoidal"
    attention_scale: float = None
    norm_first: bool = True         # FIX: pre-LN (post-LN + lr 1e-3 + no warm-up is unstable)
    n_experts:  int   = 4
    top_k:      int   = 2
    aux_loss_coef: float = 0.01
    attention_type: str = "gqa"
    use_moe: bool = False, 
    use_conv: bool = False
    local_kernel_size: int = 25
    
# Rotary Positional Embeddings
def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """ Returns (cos, sin) : the split-half rotate_half convention: channel i is paired with channel i+d_head/2. """
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    ang = torch.cat([torch.outer(torch.arange(max_seq_len).float(), inv)] * 2, -1)
    return ang.cos(), ang.sin()
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree rotation step used by the "rotate_half" formulation of RoPE."""
    x1, x2 = x[ ..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 : ]
    return torch.cat((-x2, x1), dim = -1)
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """ x: (B, n_heads, T, d_head) -- a query or key tensor. cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T. Returns the rotated tensor, same shape as x. Formula: x_rotated = x * cos + rotate_half(x) * sin """
    cos, sin = (cos[None, None], sin[None, None]) if cos.dim() == 2 else (cos[:, None], sin[:, None])
    return x.float() * cos + rotate_half(x.float()) * sin

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)
    angle = pos / (10000 ** (i / d_model))
    return torch.stack([torch.sin(angle), torch.cos(angle)], dim=-1).reshape(max_len, d_model)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=True, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep, self.d_head = n_heads // n_kv_heads, d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[position_ids], self.rope_sin[position_ids])
            k = apply_rope(k, self.rope_cos[position_ids], self.rope_sin[position_ids])
             
        # Append to KV cache if provided
        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        new_kv_cache = (k, v) if use_cache else None

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # Only set is_causal=True if processing multiple prompt tokens at once (T > 1)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0,  is_causal=is_causal, scale=self.attention_scale)
        out = self.resid_dropout(self.out_proj(context.transpose(1, 2).reshape(B, T, C)))
        
        return (out, new_kv_cache) if use_cache else out

class MultiHeadLatentAttention(nn.Module):
    """Joint KV Compression Multi-Head Latent Attention module drop-in replacement for CausalSelfAttention."""

    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, dropout=0.0, attention_scale=None):
        super().__init__()
        assert (d_model % n_heads == 0 and 0 < kv_latent_dim < d_model), "d_model must be divisible by n_heads, and 0 < kv_latent_dim < d_model"

        self.n_heads, self.kv_latent_dim, self.d_head = n_heads, kv_latent_dim, d_model // n_heads
        self.attention_scale = ((1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale))

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_down_proj = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.k_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        # Dropouts
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, position_ids=None, attn_mask=None, kv_cache=None, use_cache=False):
        """Forward pass supporting sequence generation with KV cache and FlashAttention/SDPA."""
        B, T, C = x.shape
        q = (self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T, d_head)
        c_kv = self.kv_down_proj(x)  # (B, T, kv_latent_dim)

        if kv_cache is not None:
            past_c_kv = (kv_cache[0] if isinstance(kv_cache, (tuple, list)) else kv_cache)
            c_kv = torch.cat([past_c_kv, c_kv], dim=1)  # Append along sequence dim T
        new_kv_cache = (c_kv,) if use_cache else None

        k = (self.k_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        v = (self.v_up_proj(c_kv).view(B, -1, self.n_heads, self.d_head).transpose(1, 2))  # (B, n_heads, T_total, d_head)
        is_causal = (attn_mask is None) and (kv_cache is None) and (T > 1)
        
        context = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=is_causal, scale=self.attention_scale)
        out = context.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        return (out, new_kv_cache) if use_cache else out

### Model Definitions
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)
        
class SwiGLUFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up   = nn.Linear(d_model, d_ff, bias=False); self.down = nn.Linear(d_ff, d_model, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class TopKRouter(nn.Module):
    def __init__(self, d_model: int, n_experts: int, top_k: int):
        super().__init__()
        self.top_k  = top_k; self.linear = nn.Linear(d_model, n_experts, bias=False)
    def forward(self, x_flat):                           
        logits = self.linear(x_flat); topk_vals, topk_idx = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(topk_vals, dim=-1)            # (B*T, top_k)
        dispatch = torch.zeros_like(logits, dtype=weights.dtype)
        dispatch.scatter_(-1, topk_idx, weights)          # (B*T, n_experts)
        return dispatch, logits                           # logits kept for aux loss

class MoEFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k, dropout, aux_coef=0.01):
        super().__init__()
        self.router   = TopKRouter(d_model, n_experts, top_k)
        self.experts  = nn.ModuleList([SwiGLUFeedForward(d_model, d_ff, dropout) for _ in range(n_experts)])
        self.aux_coef = aux_coef; self.aux_loss = torch.tensor(0.0)

    def forward(self, x):
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)
        dispatch, router_logits = self.router(x_flat)

        z = torch.logsumexp(router_logits, dim=-1)
        self.aux_loss = self.aux_coef * (z ** 2).mean()
        out = torch.zeros_like(x_flat)
        for e_idx, expert in enumerate(self.experts):
            w  = dispatch[:, e_idx];  mask = w > 0
            if not mask.any():
                continue
            out[mask] += w[mask].unsqueeze(-1) * expert(x_flat[mask])

        return out.reshape(B, T, D)

class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = cfg.norm_first; self.ln1 = nn.LayerNorm(cfg.d_model); self.ln2 = nn.LayerNorm(cfg.d_model)
        if(cfg.attention_type == "mla"):
            self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        else:
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"), attention_scale=cfg.attention_scale)
        if(cfg.use_moe):
            self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        else:
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        
    def _attn(self, h, position_ids, attn_mask, kv_cache, use_cache):
        r = self.attn(h, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        return r if use_cache else (r, None)
        
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        if self.norm_first:
            a, new_cache = self._attn(self.ln1(x), position_ids, attn_mask, kv_cache, use_cache)
            x = x + a
            x = x + self.ffn(self.ln2(x))
        else:
            a, new_cache = self._attn(x, position_ids, attn_mask, kv_cache, use_cache)
            x = self.ln1(x + a)
            x = self.ln2(x + self.ffn(x))
        return x, new_cache
        
class LocalDepthwiseConv(nn.Module):
    """ Depthwise causal convolution over the sequence dimension. Acts on each channel independently — O(T * C * kernel) cost. kernel_size = 25 covers ~25 tokens of local context. """
    def __init__(self, d_model: int, kernel_size: int = 25, dropout: float = 0.0):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv1d(in_channels=d_model, out_channels=d_model, kernel_size=kernel_size, padding=0, groups=d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        nn.init.normal_(self.conv.weight, 0.0, 0.02)
        nn.init.normal_(self.proj.weight, 0.0, 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, C) → (B, T, C)"""
        B, T, C = x.shape
        xc = x.transpose(1, 2)
        xc = F.pad(xc, (self.kernel_size - 1, 0))
        xc = self.conv(xc)          # (B, C, T)  — output length = T (causal)
        xc = xc.transpose(1, 2)    # (B, T, C)
        return self.drop(self.proj(F.gelu(xc)))

class ParallelLocalGlobalBlock(nn.Module):
    """Pre-LN block with global attention and local conv running IN PARALLEL."""
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.RMSNorm(cfg.d_model); self.ln2  = nn.RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, use_rope=cfg.pos_emb_type == "rope", attention_scale=cfg.attention_scale)
        self.conv = LocalDepthwiseConv(cfg.d_model, kernel_size=cfg.local_kernel_size, dropout=cfg.dropout)
        self.ffn  = MoEFeedForward(cfg.d_model, cfg.d_ff, cfg.n_experts, cfg.top_k, cfg.dropout, cfg.aux_loss_coef)
        self.alpha = nn.Parameter(torch.ones(1))    # global attention scale
        self.beta  = nn.Parameter(torch.full((1,), 0.1))  # local conv scale

    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        normed = self.ln1(x)
        attn_out = self.attn(normed, position_ids=position_ids, attn_mask=attn_mask, kv_cache=kv_cache, use_cache=use_cache)
        attn_out, new_cache = attn_out if use_cache else attn_out, None
        conv_out = self.conv(normed)
        
        x = x + self.alpha * attn_out + self.beta * conv_out
        x = x + self.ffn(self.ln2(x))
        return x, new_cache

    @property
    def aux_loss(self):
        return self.ffn.aux_loss


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.use_sin = cfg.pos_emb_type == "sinusoidal"
        self.emb_scale = math.sqrt(cfg.d_model) if self.use_sin else 1.0
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if self.use_sin:
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        if cfg.use_conv:
            self.blocks = nn.ModuleList([ParallelLocalGlobalBlock(cfg) for _ in range(cfg.n_layers)])
        else:
            self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self._init_weights()
        self.head.weight = self.tok_emb.weight # tie AFTER init so the embedding init survives

    def aux_loss(self):
        return sum(blk.ffn.aux_loss for blk in self.blocks)
        
    def _init_weights(self):
        emb_std = self.cfg.d_model ** -0.5 if self.use_sin else 0.02
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=emb_std)
        for n, p in self.named_parameters():
            if n.endswith("attn.out_proj.weight") or n.endswith("ffn.net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.cfg.n_layers))
    
    def forward(self, idx: torch.Tensor, kv_cache_list=None, use_cache=False, attention_mask=None) -> torch.Tensor:
        """attention_mask: optional (B, past+T) tensor, 1 = real token, 0 = left padding."""
        B, T = idx.shape
        past = kv_cache_list[0][0].shape[-2] if kv_cache_list is not None else 0
        S = past + T

        if attention_mask is None:
            position_ids = torch.arange(past, S, device=idx.device).unsqueeze(0).expand(B, T)
            attn_mask = None                                   # fast path: is_causal=True inside SDPA
        else:
            position_ids = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)[:, past:S]
            q_idx = torch.arange(past, S, device=idx.device)[:, None]
            k_idx = torch.arange(S, device=idx.device)[None, :]
            allowed = (k_idx <= q_idx)[None] & attention_mask[:, None, :S].bool()   # causal & not-padding
            allowed = allowed | (k_idx == q_idx)[None]         # pad rows attend to themselves (no NaNs)
            attn_mask = allowed.unsqueeze(1)                   # (B, 1, T, S)

        x = self.tok_emb(idx) * self.emb_scale
        if self.use_sin:
            x = x + self.pos_emb[position_ids]

        new_caches = []
        for i, block in enumerate(self.blocks):
            layer_cache = kv_cache_list[i] if kv_cache_list is not None else None
            x, c = block(x, position_ids, attn_mask, layer_cache, use_cache)
            new_caches.append(c)

        logits = self.head(self.ln_f(x))
        return (logits, new_caches) if use_cache else logits
        
    @torch.no_grad()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=384, temperature=0.7, top_p=0.9,
                 repetition_penalty=1.0, banned_token_ids=(0, 1, 2, 3), do_sample=True):
        self.eval()
        out = input_ids
        if attention_mask is None:
            attention_mask = torch.ones_like(out)
    
        logits, kv = self(out, use_cache=True, attention_mask=attention_mask)      # prefill
        next_logits = logits[:, -1, :].float()
    
        for step in range(max_new_tokens):
            l = next_logits.clone()
            if repetition_penalty != 1.0:
                sc = l.gather(1, out); sc = torch.where(sc < 0, sc * repetition_penalty, sc / repetition_penalty)
                l = l.scatter(1, out, sc)
            if banned_token_ids:                       # never emit <pad>/<unk>/<s>/</s>: we need 128 words
                l[:, list(banned_token_ids)] = float("-inf")

            if do_sample and temperature > 0.0:
                l = l / temperature
                if top_p < 1.0:
                    sorted_logits, sorted_idx = torch.sort(l, descending=True)
                    cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    remove = cum > top_p
                    remove[..., 1:] = remove[..., :-1].clone()
                    remove[..., 0] = False
                    l = l.masked_fill(remove.scatter(1, sorted_idx, remove), float("-inf"))
                next_token = torch.multinomial(F.softmax(l, dim=-1), num_samples=1)
            else:
                next_token = l.argmax(dim=-1, keepdim=True)

            out = torch.cat([out, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
            if step == max_new_tokens - 1:
                break
            logits, kv = self(next_token, kv_cache_list=kv, use_cache=True, attention_mask=attention_mask)
            next_logits = logits[:, -1, :].float()
        return out

## 4. Data Loading

In [20]:
def load_phase1_model(model_name):
    cfg = LMConfig()
    for attr, val in PHASE1_CONFIGS[model_name].items():
        setattr(cfg, attr, val)

    checkpoint_data = torch.load(f"./base-models/{model_name}.pt", map_location="cpu", weights_only=False)
    base_lm = DecoderLM(cfg)
    base_lm.load_state_dict(checkpoint_data["model"], strict=True)
    del checkpoint_data; gc.collect()
    return base_lm

@dataclass
class CaseSplit:
    ids: list                    # case ids, in file order
    texts: list                  # cleaned case text
    fingerprints: list           # md5 of cleaned text (dedup + token cache key)
    labels: np.ndarray           # None for test
    token_ids: list              # filled by the tokenization cell

def clean_text(text):
    """Light cleanup. Keep consistent with whatever was applied to the Phase 1 pretraining text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)          # collapse runs of spaces/tabs
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse many blank lines
    return text.strip()

def fingerprint(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def read_split(relative_path):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=f"{PHASE4_DIR}/{relative_path}")
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    assert isinstance(records, list), f"{relative_path}: expected a list of records, got {type(records)}"
    return records

def build_split(records, has_labels, split_name):
    ids = [r[ID_KEY] for r in records]
    # assert len(set(ids)) == len(ids), f"{split_name}: duplicate case ids"
    assert all(isinstance(r[TEXT_KEY], str) for r in records), f"{split_name}: non-string text"
    texts = [clean_text(r[TEXT_KEY]) for r in records]
    assert all(len(t) > 0 for t in texts), f"{split_name}: empty text after cleaning"
    labels = None
    if has_labels:
        labels = np.array([int(r[LABEL_KEY]) for r in records])
        assert set(labels.tolist()) <= {0, 1}, f"{split_name}: labels must be 0/1"
    return CaseSplit(ids, texts, [fingerprint(t) for t in texts], labels, None)

def select_rows(split, rows):
    rows = list(rows)
    return CaseSplit(
        ids=[split.ids[i] for i in rows],
        texts=[split.texts[i] for i in rows],
        fingerprints=[split.fingerprints[i] for i in rows],
        labels=None if split.labels is None else split.labels[rows],
        token_ids=None if split.token_ids is None else [split.token_ids[i] for i in rows])

train_split = build_split(read_split("Train/train.json"), True,  "train")
dev_split   = build_split(read_split("Train/dev.json"),   True,  "dev")
test_split  = build_split(read_split("Test/test.json"),   False, "test")
print("raw sizes -> train", len(train_split.ids), "| dev", len(dev_split.ids), "| test", len(test_split.ids))

# 1) exact duplicate texts inside train: keep the first occurrence, report label conflicts
first_row, keep_rows, conflicts = {}, [], 0
for row, (fp, label) in enumerate(zip(train_split.fingerprints, train_split.labels)):
    if fp in first_row:
        conflicts += int(train_split.labels[first_row[fp]] != label)
    else:
        first_row[fp] = row; keep_rows.append(row)
print(f"train: removed {len(train_split.ids) - len(keep_rows)} duplicate texts ({conflicts} with conflicting labels)")
train_split = select_rows(train_split, keep_rows)

# 2) dev cases whose text also appears in train would inflate model selection -> drop them
train_fps = set(train_split.fingerprints)
dev_keep = [i for i, fp in enumerate(dev_split.fingerprints) if fp not in train_fps]
print(f"dev: removed {len(dev_split.ids) - len(dev_keep)} cases that also occur in train")
dev_split = select_rows(dev_split, dev_keep)

# 3) test is never filtered (one prediction per test id is required); only report overlap
print("test texts also present in train:", sum(fp in train_fps for fp in test_split.fingerprints))
print("id overlap train/dev:", len(set(train_split.ids) & set(dev_split.ids)),
      "| train/test:", len(set(train_split.ids) & set(test_split.ids)))

if SMOKE_TEST:
    rng = np.random.RandomState(SEED)
    train_split = select_rows(train_split, rng.choice(len(train_split.ids), 3000, replace=False))
    dev_split   = select_rows(dev_split,   rng.choice(len(dev_split.ids),   100, replace=False))
    test_split  = select_rows(test_split,  rng.choice(len(test_split.ids),   200, replace=False))

raw sizes -> train 37387 | dev 1988 | test 1517
train: removed 5734 duplicate texts (0 with conflicting labels)
dev: removed 0 cases that also occur in train
test texts also present in train: 0
id overlap train/dev: 0 | train/test: 0


In [21]:
TOKEN_CACHE_PATH = f"{OUT}/token_cache_tokenizerv3_{VOCAB_SIZE}.pkl"
token_cache = pickle.load(open(TOKEN_CACHE_PATH, "rb")) if os.path.exists(TOKEN_CACHE_PATH) else {}

def tokenize_split(split, split_name):
    """Token ids of the whole case WITHOUT <s>/</s> (each chunk gets its own <s>)."""
    for text, fp in tqdm(zip(split.texts, split.fingerprints), total=len(split.texts), desc=f"tokenize {split_name}"):
        if fp not in token_cache:
            token_cache[fp] = tokenizer.encode(text)[1:-1]
    split.token_ids = [token_cache[fp] for fp in split.fingerprints]

for name, split in [("train", train_split), ("dev", dev_split), ("test", test_split)]:
    tokenize_split(split, name)
pickle.dump(token_cache, open(TOKEN_CACHE_PATH, "wb"))

# sanity: byte-level BPE should almost never emit <unk>
all_tokens = np.concatenate([np.array(t) for t in train_split.token_ids])
print("unk fraction in train tokens:", (all_tokens == UNK).mean())


# ── EDA: look at the data before training ──
def chunk_starts(num_tokens):
    """Start offsets of overlapping windows; the last window is aligned to the end of the document."""
    if num_tokens <= CONTENT_LEN:
        return [0]
    starts = list(range(0, num_tokens - CONTENT_LEN + 1, CHUNK_STRIDE))
    if starts[-1] + CONTENT_LEN < num_tokens:
        starts.append(num_tokens - CONTENT_LEN)
    return starts

print("record keys:", sorted(train_split.__dict__.keys()))
for name, split in [("train", train_split), ("dev", dev_split)]:
    print(f"{name}: n={len(split.ids)} | label counts {dict(Counter(split.labels.tolist()))} | positive rate {split.labels.mean():.3f}")

train_lengths = np.array([len(t) for t in train_split.token_ids])
train_chunks  = np.array([len(chunk_starts(n)) for n in train_lengths])
print("tokens/case percentiles (50/75/90/95/99/max):", np.percentile(train_lengths, [50, 75, 90, 95, 99, 100]).astype(int))
print("chunks/case percentiles (50/75/90/95/99/max):", np.percentile(train_chunks,  [50, 75, 90, 95, 99, 100]).astype(int))
print(f"cases that fit entirely in MAX_CHUNKS={MAX_CHUNKS}: {(train_chunks <= MAX_CHUNKS).mean():.1%}")
for label in (0, 1):
    print(f"mean tokens, label {label}: {train_lengths[train_split.labels == label].mean():.0f}")

# Leakage eyeball check: the END of a case is where outcome phrases would hide
for label in (0, 1):
    for i in np.where(train_split.labels == label)[0][:2]:
        print(f"\n--- label {label} | id {train_split.ids[i]} | last 300 chars ---\n{train_split.texts[i][-300:]}")

tokenize test: 100%|██████████| 200/200 [00:00<00:00, 298.62it/s]


unk fraction in train tokens: 0.0
record keys: ['fingerprints', 'ids', 'labels', 'texts', 'token_ids']
train: n=3000 | label counts {0: 1744, 1: 1256} | positive rate 0.419
dev: n=100 | label counts {1: 54, 0: 46} | positive rate 0.540
tokens/case percentiles (50/75/90/95/99/max): [  2445   4438   7642  10673  21017 148931]
chunks/case percentiles (50/75/90/95/99/max): [  6  10  17  24  47 334]
cases that fit entirely in MAX_CHUNKS=8: 66.2%
mean tokens, label 0: 3779
mean tokens, label 1: 3480

--- label 0 | id 1970_322 | last 300 chars ---
 only exceeded their right of private defence is equally unfounded because on the facts on the record and on companycurrent companyclusions of the two Courts below there is numberquestion of any right of private defence because the prosecution party has number at all been shown to be the aggressOrs.

--- label 0 | id 1997_906 | last 300 chars ---
ness cannot be regarded as are liable and independent witness in view of the material improvement made b

In [22]:
REGION_THIRDS = {"first_third": 0, "middle_third": 1, "last_third": 2}   # used for the section probe

def pick_evenly(candidates, k):
    if len(candidates) <= k:
        return candidates
    positions = np.unique(np.linspace(0, len(candidates) - 1, k).round().astype(int))
    return [candidates[p] for p in positions]

def select_chunk_indices(num_chunks, strategy, max_chunks):
    """Which chunks of a case are fed to the model."""
    all_idx = list(range(num_chunks))
    if strategy == "head":
        return all_idx[:max_chunks]
    if strategy == "tail":
        return all_idx[-max_chunks:]
    if strategy == "head_tail":                                  # beginning (facts) + end (reasoning)
        if num_chunks <= max_chunks:
            return all_idx
        n_head = max_chunks // 2
        return all_idx[:n_head] + all_idx[-(max_chunks - n_head):]
    if strategy == "uniform":                                    # evenly spaced over the whole case
        return pick_evenly(all_idx, max_chunks)
    if strategy in REGION_THIRDS:                                # only one third of the case
        third = REGION_THIRDS[strategy]
        lo = num_chunks * third // 3
        hi = max(num_chunks * (third + 1) // 3, lo + 1)
        return pick_evenly(all_idx[lo:hi], max_chunks)
    raise ValueError(f"unknown chunk strategy: {strategy}")

class CaseDataset(Dataset):
    """One item = one case = a few chunks (+ label, + optional teacher logits)."""
    def __init__(self, split, strategy, max_chunks):
        self.split, self.strategy, self.max_chunks = split, strategy, max_chunks
        self.teacher_logits = None                              # set for the KD student

    def __len__(self):
        return len(self.split.ids)

    def __getitem__(self, i):
        token_ids = self.split.token_ids[i]
        starts = chunk_starts(len(token_ids))
        chosen = select_chunk_indices(len(starts), self.strategy, self.max_chunks)
        item = {"chunks": [[BOS] + token_ids[starts[c]: starts[c] + CONTENT_LEN] for c in chosen]}
        if self.split.labels is not None:
            item["label"] = int(self.split.labels[i])
        if self.teacher_logits is not None:
            item["teacher_logits"] = self.teacher_logits[i]
        return item

def collate_cases(items):
    """Pad to ids [B, max_chunks_in_batch, max_len] and chunk_mask [B, max_chunks_in_batch]."""
    n_chunks = max(len(it["chunks"]) for it in items)
    max_len  = max(len(c) for it in items for c in it["chunks"])
    ids = torch.full((len(items), n_chunks, max_len), PAD, dtype=torch.long)
    chunk_mask = torch.zeros(len(items), n_chunks, dtype=torch.bool)
    for b, it in enumerate(items):
        for c, chunk in enumerate(it["chunks"]):
            ids[b, c, :len(chunk)] = torch.tensor(chunk)
            chunk_mask[b, c] = True
    batch = {"ids": ids, "chunk_mask": chunk_mask}
    if "label" in items[0]:
        batch["labels"] = torch.tensor([it["label"] for it in items])
    if "teacher_logits" in items[0]:
        batch["teacher_logits"] = torch.stack([it["teacher_logits"] for it in items])
    return batch

## 5. Finetuning methods

In [23]:
class LoRALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A x)   (additive low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """y = Wx + (alpha/r) * B(A(Wx))  ==  (I + s*BA) W x   (multiplicative low-rank update)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Parameter(torch.empty(rank, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

class AdaLoRALinear(nn.Module):
    """Delta W = P diag(lambda * mask) Q. Orthogonality penalty + rank pruning (see AdaLoRAController)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.rank, self.scale = base, rank, alpha / rank
        self.P    = nn.Parameter(torch.empty(base.out_features, rank))
        self.diag = nn.Parameter(torch.zeros(rank))
        self.Q    = nn.Parameter(torch.empty(rank, base.in_features))
        self.register_buffer("mask", torch.ones(rank))           # 0 = pruned singular value
        nn.init.orthogonal_(self.P); nn.init.orthogonal_(self.Q)
    def orth_loss(self):
        eye = torch.eye(self.rank, device=self.P.device)
        return ((self.P.T @ self.P - eye) ** 2).sum() + ((self.Q @ self.Q.T - eye) ** 2).sum()
    def forward(self, x):
        low_rank = F.linear(x, self.Q) * (self.diag * self.mask)
        return self.base(x) + self.scale * F.linear(low_rank, self.P)

class AdaLoRAController:
    """Prunes singular values by smoothed sensitivity |lambda * grad| following a cubic budget schedule."""
    def __init__(self, modules, total_steps, final_ratio, warmup_frac=0.1, freeze_frac=0.2, prune_every=20, beta=0.85):
        self.modules, self.prune_every, self.beta = modules, prune_every, beta
        self.initial_budget = sum(m.rank for m in modules)
        self.final_budget = int(self.initial_budget * final_ratio)
        self.start_step = int(total_steps * warmup_frac)
        self.end_step = int(total_steps * (1 - freeze_frac))
        self.importance = [torch.zeros_like(m.diag) for m in modules]
    def budget_at(self, step):
        if step < self.start_step: return self.initial_budget
        if step >= self.end_step:  return self.final_budget
        progress = (step - self.start_step) / (self.end_step - self.start_step)
        return int(self.final_budget + (self.initial_budget - self.final_budget) * (1 - progress) ** 3)
    @torch.no_grad()
    def update(self, step):
        """Call after backward (grads unscaled), before optimizer.step()."""
        if step >= self.end_step:
            return
        for module, imp in zip(self.modules, self.importance):
            imp.mul_(self.beta).add_((1 - self.beta) * (module.diag * module.diag.grad).abs())
        if step >= self.start_step and step % self.prune_every == 0:
            all_scores = torch.cat(self.importance)
            budget = self.budget_at(step)
            threshold = torch.topk(all_scores, budget).values[-1]
            for module, imp in zip(self.modules, self.importance):
                module.mask.copy_((imp >= threshold).float())
    def active_rank(self):
        return int(sum(m.mask.sum().item() for m in self.modules))

class AdapterBlock(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a transformer block."""
    def __init__(self, block, d_model, bottleneck):
        super().__init__()
        self.block = block
        self.down, self.up = nn.Linear(d_model, bottleneck), nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x, position_ids, attn_mask=None, kv_cache=None, use_cache=False):
        h, cache = self.block(x, position_ids, attn_mask)
        return h + self.up(F.gelu(self.down(h))), cache

PEFT_WRAPPERS = {"lora": LoRALinear, "lorma": LoRMALinear, "adalora": AdaLoRALinear}
ATTN_PROJ_NAMES = ("q_proj", "k_proj", "v_proj", "out_proj", "kv_down_proj", "k_up_proj", "v_up_proj")  # GQA and MLA names

def wrap_attention_projections(backbone, wrapper_cls, rank, alpha):
    """Wrap every attention projection that exists in this architecture (GQA / MLA / conv variants)."""
    for block in backbone.blocks:
        for name in ATTN_PROJ_NAMES:
            if hasattr(block.attn, name):
                setattr(block.attn, name, wrapper_cls(getattr(block.attn, name), rank, alpha))

## 6. Training utilities

In [24]:
def encode_chunks(backbone, ids, use_ckpt):
    """Phase 1 decoder over [N, L] chunks (causal, right-padded) -> hidden states [N, L, D].
    Right padding + causal attention means padding never influences real tokens, so no mask is needed."""
    N, L = ids.shape
    position_ids = torch.arange(L, device=ids.device).unsqueeze(0).expand(N, L)
    x = backbone.tok_emb(ids) * backbone.emb_scale
    if backbone.use_sin:
        x = x + backbone.pos_emb[position_ids]
    for block in backbone.blocks:
        if use_ckpt and torch.is_grad_enabled():
            x = checkpoint(lambda h, blk=block: blk(h, position_ids, None)[0], x, use_reentrant=False)
        else:
            x = block(x, position_ids, None)[0]
    return backbone.ln_f(x)

# ── pooling of chunk vectors [B, C, D] into one case vector [B, D] ──
class MeanPooler(nn.Module):
    def forward(self, grid, mask):
        m = mask.unsqueeze(-1).to(grid.dtype)
        return (grid * m).sum(1) / m.sum(1)

class AttentionPooler(nn.Module):
    """Learned attention over chunks."""
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)
    def forward(self, grid, mask):
        scores = self.score(grid).squeeze(-1).float().masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return (weights.unsqueeze(-1) * grid.float()).sum(1)

class BiGRUAttentionPooler(nn.Module):
    """Hierarchical: BiGRU over the ordered chunk sequence (section-aware), then attention pooling."""
    def __init__(self, dim):
        super().__init__()
        self.gru = nn.GRU(dim, dim // 2, batch_first=True, bidirectional=True)
        self.attn = AttentionPooler(dim)
    def forward(self, grid, mask):
        packed = pack_padded_sequence(grid.float(), mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=grid.size(1))
        return self.attn(grid.float() + out, mask)

POOLERS = {"mean": lambda d: MeanPooler(), "attn": AttentionPooler, "bigru": BiGRUAttentionPooler}

class CaseClassifier(nn.Module):
    def __init__(self, backbone, pooling, use_ckpt):
        super().__init__()
        dim = backbone.cfg.d_model
        self.backbone, self.use_ckpt = backbone, use_ckpt
        self.doc_pooler = POOLERS[pooling](dim)
        self.dropout = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(dim, NUM_LABELS)
        nn.init.normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)

    def forward(self, ids, chunk_mask):
        B, C, L = ids.shape
        flat_ids = ids[chunk_mask]                                   # only real chunks: [N, L]
        hidden = encode_chunks(self.backbone, flat_ids, self.use_ckpt)
        token_w = flat_ids.ne(PAD).unsqueeze(-1).to(hidden.dtype)
        chunk_repr = (hidden * token_w).sum(1) / token_w.sum(1).clamp(min=1.0)       # masked mean over tokens
        grid = chunk_repr.new_zeros(B, C, chunk_repr.size(-1))
        grid[chunk_mask] = chunk_repr                                # back to [B, C, D]
        return self.classifier(self.dropout(self.doc_pooler(grid, chunk_mask)))

# ── model factory ──
def freeze(module):
    for p in module.parameters():
        p.requires_grad = False

def cast_frozen_linears_to_half(model):
    """Store frozen nn.Linear weights in fp16 once, instead of letting autocast re-cast them on every call."""
    skip = {id(m) for m in model.backbone.head.modules()} | {id(model.backbone.head)}   # tied with the embedding
    n_cast = 0
    for module in model.modules():
        if isinstance(module, nn.Linear) and id(module) not in skip and not any(p.requires_grad for p in module.parameters()):
            module.half(); n_cast += 1
    print(f"cast {n_cast} frozen Linear layers to fp16")
    return model

def build_model(base_lm, exp):
    backbone = copy.deepcopy(base_lm)
    if exp.block_stride > 1:                                         # KD student: keep every n-th block
        keep = list(range(0, len(backbone.blocks), exp.block_stride))
        backbone.blocks = nn.ModuleList([backbone.blocks[i] for i in keep])
    d_model = backbone.cfg.d_model

    if exp.method in ("full", "unfreeze"):
        pass                                                         # everything trainable ("unfreeze" schedules it per epoch)
    elif exp.method == "partial":                                    # top-n blocks + final norm
        freeze(backbone)
        for block in backbone.blocks[-exp.n_top:]:
            for p in block.parameters(): p.requires_grad = True
        for p in backbone.ln_f.parameters(): p.requires_grad = True
    elif exp.method == "bitfit":                                     # bias terms only
        for name, p in backbone.named_parameters():
            p.requires_grad = name.endswith(".bias")
        assert any(p.requires_grad for p in backbone.parameters()), f"{exp.base_model} has no bias terms for BitFit"
    elif exp.method in PEFT_WRAPPERS:
        freeze(backbone)
        wrap_attention_projections(backbone, PEFT_WRAPPERS[exp.method], exp.rank, exp.alpha)
    elif exp.method == "adapter":
        freeze(backbone)
        backbone.blocks = nn.ModuleList([AdapterBlock(b, d_model, exp.adapter_dim) for b in backbone.blocks])
    else:
        raise ValueError(f"unknown method: {exp.method}")
    return CaseClassifier(backbone, exp.pooling, GRAD_CKPT).to(DEVICE)

def count_params(model):
    total = sum(p.numel() for p in model.parameters()) / 1e6
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    return total, trainable

def apply_gradual_unfreezing(model, epoch, total_epochs):
    """Layer-unfreezing schedule: top blocks first, one more group of lower blocks every epoch."""
    blocks = model.backbone.blocks
    n_unfrozen = math.ceil(len(blocks) * (epoch + 1) / total_epochs)
    for i, block in enumerate(blocks):
        for p in block.parameters():
            p.requires_grad = i >= len(blocks) - n_unfrozen
    for p in model.backbone.ln_f.parameters(): p.requires_grad = True
    for p in model.backbone.tok_emb.parameters(): p.requires_grad = (n_unfrozen == len(blocks))

In [25]:
def balanced_class_weights(labels):
    counts = np.bincount(labels, minlength=NUM_LABELS)
    return torch.tensor(len(labels) / (NUM_LABELS * counts), dtype=torch.float32, device=DEVICE)

def compute_loss(logits, labels, class_weights, loss_type):
    if loss_type == "ce":                                            # no imbalance handling
        return F.cross_entropy(logits, labels)
    if loss_type == "weighted_ce":                                   # inverse-frequency class weights
        return F.cross_entropy(logits, labels, weight=class_weights)
    if loss_type == "focal":                                         # down-weights easy cases + class weights
        weighted_ce = F.cross_entropy(logits, labels, weight=class_weights, reduction="none")
        pt = torch.exp(-F.cross_entropy(logits, labels, reduction="none"))
        return (((1 - pt) ** FOCAL_GAMMA) * weighted_ce).mean()
    raise ValueError(loss_type)

def evaluate_probs(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    return {"F1": f1_score(labels, preds, average="macro"),
            "P":  precision_score(labels, preds, average="macro", zero_division=0),
            "R":  recall_score(labels, preds, average="macro", zero_division=0),
            "F1_per_class": f1_score(labels, preds, average=None, labels=[0, 1]).tolist()}

def tune_threshold(probs, labels):
    """Calibration/post-processing: pick the probability threshold that maximises macro-F1 on dev."""
    grid = np.arange(0.05, 0.96, 0.01)
    scores = [f1_score(labels, (probs >= t).astype(int), average="macro") for t in grid]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])

@torch.no_grad()
def predict_logits(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=EVAL_BATCH_DOCS, shuffle=False, num_workers=2, collate_fn=collate_cases)
    all_logits = []
    for batch in tqdm(loader, desc="predict", leave=False):
        with autocast("cuda", dtype=torch.float16):
            logits = model(batch["ids"].to(DEVICE), batch["chunk_mask"].to(DEVICE))
        all_logits.append(logits.float().cpu())
    return torch.cat(all_logits)

def probs_from_logits(logits):
    return torch.softmax(logits, dim=-1)[:, 1].numpy()

## 7. Training Loop

In [26]:
def checkpoint_path(name):
    return f"{OUT}/ckpt_{name}.pt"     # latest epoch (resume point), overwritten every epoch

def best_path(name):
    return f"{OUT}/best_{name}.pt"     # best epoch only, kept separately

def save_atomic(obj, path):
    """Write to a temp file, then rename: an interruption can never corrupt the previous file."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}

def set_rng_state(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"]); torch.cuda.set_rng_state_all(state["cuda"])

def save_best(exp, best_state, best_metric, epoch):
    save_atomic({"state": best_state, "metric": best_metric, "epoch": epoch}, best_path(exp.name))

def save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch, global_step, history):
    """Everything needed to continue training from the end of `epoch` (epochs completed so far)."""
    model_state = {k: v.detach().cpu() for k, v in model.state_dict().items() if k in snapshot_keys}
    save_atomic({"config": vars(exp), "epoch": epoch, "global_step": global_step, "history": history,
                 "model": model_state, "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                 "adalora_importance": None if adalora_ctrl is None else [imp.cpu() for imp in adalora_ctrl.importance],
                 "rng": get_rng_state()}, checkpoint_path(exp.name))

def resume_if_checkpointed(exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric):
    """Returns (start_epoch, global_step, history, best_metric, best_state).
    Fresh start values are returned when this experiment has no checkpoint yet."""
    if not os.path.exists(checkpoint_path(exp.name)):
        return 0, 0, fresh_history, worst_metric, None

    ckpt = torch.load(checkpoint_path(exp.name), map_location="cpu", weights_only=False)
    # assert ckpt["config"] == vars(exp), f"{checkpoint_path(exp.name)} was written by a different configuration; delete it (and best_{exp.name}.pt) to start over"
    outcome = model.load_state_dict(ckpt["model"], strict=False)
    assert not outcome.unexpected_keys, outcome.unexpected_keys
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])
    if adalora_ctrl is not None:
        for importance, saved in zip(adalora_ctrl.importance, ckpt["adalora_importance"]):
            importance.copy_(saved)
    set_rng_state(ckpt["rng"])

    best = torch.load(best_path(exp.name), map_location="cpu", weights_only=False)
    print(f"[{exp.name}] resumed after epoch {ckpt['epoch']}/{exp.epochs} (best so far {best['metric']:.4f} at epoch {best['epoch']})")
    return ckpt["epoch"], ckpt["global_step"], ckpt["history"], best["metric"], best["state"]

In [27]:
def snapshot_names(model, exp):
    """Parameters that must be restored to recover a given epoch."""
    if exp.method == "unfreeze":
        return set(model.state_dict().keys())
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    return trainable | {k for k in model.state_dict() if k.endswith(".mask")}

def train_model(model, exp, train_ds, dev_ds):
    loader = DataLoader(train_ds, batch_size=exp.batch_docs, shuffle=True, collate_fn=collate_cases, num_workers=2, pin_memory=True)
    class_weights = balanced_class_weights(train_ds.split.labels)

    # backbone (adapters / unfrozen layers) and the new head get different learning rates
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    body_params = [p for n, p in trainable if n.startswith("backbone.")]
    head_params = [p for n, p in trainable if not n.startswith("backbone.")]
    optimizer = torch.optim.AdamW([{"params": body_params, "lr": exp.lr,      "weight_decay": WEIGHT_DECAY},
                                   {"params": head_params, "lr": exp.head_lr, "weight_decay": 0.0}])
    base_lrs = [exp.lr, exp.head_lr]

    steps_per_epoch = math.ceil(len(loader) / exp.grad_accum)
    total_steps = exp.epochs * steps_per_epoch
    warmup_steps = int(WARMUP_FRAC * total_steps)
    scaler = GradScaler("cuda")

    adalora_modules = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
    adalora_ctrl = AdaLoRAController(adalora_modules, total_steps, exp.adalora_final_ratio) if adalora_modules else None

    dev_labels = dev_ds.split.labels
    fresh_history = {"val_f1": [], "val_P": [], "val_R": [], "loss": [], "train_sec": 0.0}
    start_epoch, global_step, history, best_f1, best_state = resume_if_checkpointed(
        exp, model, optimizer, scaler, adalora_ctrl, fresh_history, worst_metric=-1.0)

    for epoch in range(start_epoch, exp.epochs):
        if exp.method == "unfreeze":
            apply_gradual_unfreezing(model, epoch, exp.epochs)
        t0 = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0

        for step_in_epoch, batch in enumerate(tqdm(loader, desc=f"{exp.name} ep {epoch + 1}/{exp.epochs}", leave=False)):
            ids, chunk_mask, labels = batch["ids"].to(DEVICE), batch["chunk_mask"].to(DEVICE), batch["labels"].to(DEVICE)
            with autocast("cuda", dtype=torch.float16):
                logits = model(ids, chunk_mask)
            logits = logits.float()

            loss = compute_loss(logits, labels, class_weights, exp.loss_type)
            if "teacher_logits" in batch:                            # knowledge distillation (soft targets)
                teacher_logits = batch["teacher_logits"].to(DEVICE)
                T = exp.kd_temperature
                kd_loss = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(teacher_logits / T, -1), reduction="batchmean") * T * T
                loss = exp.kd_ce_weight * loss + (1.0 - exp.kd_ce_weight) * kd_loss
            if adalora_modules:                                      # keep P, Q near-orthogonal
                loss = loss + ADALORA_ORTH_COEF * torch.stack([m.orth_loss() for m in adalora_modules]).mean()

            scaler.scale(loss / exp.grad_accum).backward()
            epoch_loss += loss.item()

            is_boundary = (step_in_epoch + 1) % exp.grad_accum == 0 or step_in_epoch == len(loader) - 1
            if is_boundary:
                # linear warm-up, then cosine decay
                if global_step < warmup_steps:
                    lr_frac = (global_step + 1) / max(1, warmup_steps)
                else:
                    progress = (global_step - warmup_steps) / max(1, total_steps - warmup_steps)
                    lr_frac = 0.5 * (1.0 + math.cos(math.pi * progress))
                for group, base_lr in zip(optimizer.param_groups, base_lrs):
                    group["lr"] = base_lr * lr_frac

                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                if adalora_ctrl is not None and torch.isfinite(grad_norm):
                    adalora_ctrl.update(global_step)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

        # ── validation: model selection by macro-F1 at threshold 0.5 ──
        dev_probs = probs_from_logits(predict_logits(model, dev_ds))
        metrics = evaluate_probs(dev_probs, dev_labels, 0.5)
        history["val_f1"].append(metrics["F1"]); history["val_P"].append(metrics["P"]); history["val_R"].append(metrics["R"])
        history["loss"].append(epoch_loss / len(loader)); history["train_sec"] += time.time() - t0
        extra = f" | active AdaLoRA rank {adalora_ctrl.active_rank()}/{adalora_ctrl.initial_budget}" if adalora_ctrl else ""
        print(f"[{exp.name}] ep {epoch + 1}/{exp.epochs} loss {history['loss'][-1]:.4f} | dev macro P {metrics['P']:.3f} R {metrics['R']:.3f} F1 {metrics['F1']:.4f}{extra}")

        # ── checkpoints: best model (only when improved) first, then the resume point ──
        snapshot_keys = snapshot_names(model, exp)
        if metrics["F1"] > best_f1:
            best_f1 = metrics["F1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in snapshot_keys}
            save_best(exp, best_state, best_f1, epoch + 1)
        save_epoch_checkpoint(exp, model, snapshot_keys, optimizer, scaler, adalora_ctrl, epoch + 1, global_step, history)

    model.load_state_dict(best_state, strict=False)                  # restore best epoch
    history["best_f1"] = best_f1
    return history

## 8. Experiment Helper

In [28]:
@dataclass
class Experiment:
    name: str
    base_model: str               # one of PHASE1_CONFIGS
    method: str                   # full | partial | unfreeze | bitfit | lora | lorma | adalora | adapter
    pooling: str                  # mean | attn | bigru
    strategy: str                 # head | tail | head_tail | uniform
    loss_type: str                # ce | weighted_ce | focal
    epochs: int
    lr: float                     # backbone / adapter learning rate
    head_lr: float                # pooler + classifier learning rate
    batch_docs: int
    grad_accum: int
    rank: int                     # LoRA / LoRMA / AdaLoRA rank
    alpha: float
    n_top: int                    # blocks trained by "partial"
    adapter_dim: int
    adalora_final_ratio: float    # fraction of AdaLoRA ranks kept after pruning
    block_stride: int             # 1 = all blocks; 2 = every 2nd block (KD student)
    teacher_name: str             # experiment whose exported logits are distilled (None = no KD)
    kd_temperature: float
    kd_ce_weight: float           # weight of the hard-label loss in the KD objective
    export_teacher_logits: bool   # save train-set logits so this model can teach a student
    run_section_probe: bool       # evaluate with only some sections of each case

BASE = Experiment(name="", base_model=MAIN_MODEL, method="lorma", pooling="mean", strategy=CHUNK_STRATEGY,
                  loss_type="weighted_ce", epochs=3, lr=2e-4, head_lr=1e-3, batch_docs=BATCH_DOCS, grad_accum=GRAD_ACCUM,
                  rank=16, alpha=32.0, n_top=4, adapter_dim=64, adalora_final_ratio=0.5, block_stride=1,
                  teacher_name=None, kd_temperature=2.0, kd_ce_weight=0.5,
                  export_teacher_logits=False, run_section_probe=False)

def make_exp(name, **overrides):
    return replace(BASE, name=name, **overrides)

# ── teacher logits (for KD) ──
def teacher_logits_path(teacher_name):
    return f"{OUT}/teacher_logits_{teacher_name}.pt"

def export_teacher_logits(model, exp):
    dataset = CaseDataset(train_split, exp.strategy, MAX_CHUNKS)
    logits = predict_logits(model, dataset)
    torch.save({"ids": train_split.ids, "logits": logits, "strategy": exp.strategy, "max_chunks": MAX_CHUNKS},
               teacher_logits_path(exp.name))

def load_teacher_logits(exp):
    saved = torch.load(teacher_logits_path(exp.teacher_name), weights_only=False)
    assert saved["ids"] == train_split.ids, "teacher logits were exported for a different train set"
    assert saved["strategy"] == exp.strategy and saved["max_chunks"] == MAX_CHUNKS, "student must see the same chunks as the teacher"
    return saved["logits"]

# ── submission ──
def write_submission(preds, name):
    submission = [{"id": case_id, "pred": int(p)} for case_id, p in zip(test_split.ids, preds)]
    assert [s["id"] for s in submission] == test_split.ids
    assert all(s["pred"] in (0, 1) for s in submission)
    json_path, zip_path = f"{OUT}/submission_{name}.json", f"{OUT}/submission_{name}.zip"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, "prediction.json")
    print(f"  -> {zip_path}")

PROBE_STRATEGIES = ["head", "tail", "head_tail", "uniform", "first_third", "middle_third", "last_third"]

def run_experiment(exp):
    """Build -> train -> evaluate -> write submission -> save results. Skips finished experiments."""
    result_path = f"{OUT}/result_{exp.name}.json"
    # if os.path.exists(result_path):
    #     print(f"[{exp.name}] finished earlier -> skipped"); return

    set_seed(SEED)
    base_lm = load_phase1_model(exp.base_model)
    model = build_model(base_lm, exp)
    model = cast_frozen_linears_to_half(model)
    del base_lm; gc.collect()

    train_ds = CaseDataset(train_split, exp.strategy, MAX_CHUNKS)
    dev_ds   = CaseDataset(dev_split,   exp.strategy, MAX_CHUNKS)
    test_ds  = CaseDataset(test_split,  exp.strategy, MAX_CHUNKS)
    if exp.teacher_name is not None:
        train_ds.teacher_logits = load_teacher_logits(exp)

    torch.cuda.reset_peak_memory_stats()
    history = train_model(model, exp, train_ds, dev_ds)
    total_m, trainable_m = count_params(model)

    # dev inference (timed), threshold tuning, test predictions
    torch.cuda.synchronize(); t0 = time.perf_counter()
    dev_probs = probs_from_logits(predict_logits(model, dev_ds))
    torch.cuda.synchronize(); infer_sec_per_case = (time.perf_counter() - t0) / len(dev_ds)
    peak_gb = torch.cuda.max_memory_allocated() / 2**30

    dev_labels = dev_split.labels
    threshold_tuned, f1_tuned = tune_threshold(dev_probs, dev_labels)
    threshold = threshold_tuned if USE_TUNED_THRESHOLD else 0.5
    metrics_default = evaluate_probs(dev_probs, dev_labels, 0.5)
    metrics_final   = evaluate_probs(dev_probs, dev_labels, threshold)

    test_probs = probs_from_logits(predict_logits(model, test_ds))
    np.save(f"{OUT}/dev_probs_{exp.name}.npy", dev_probs); np.save(f"{OUT}/test_probs_{exp.name}.npy", test_probs)
    write_submission((test_probs >= threshold).astype(int), exp.name)

    section_probe = None
    if exp.run_section_probe:                                        # which part of a case carries the signal?
        section_probe = {}
        for strategy in PROBE_STRATEGIES:
            probe_probs = probs_from_logits(predict_logits(model, CaseDataset(dev_split, strategy, MAX_CHUNKS)))
            section_probe[strategy] = evaluate_probs(probe_probs, dev_labels, 0.5)["F1"]

    if exp.export_teacher_logits:
        export_teacher_logits(model, exp)

    result = {"name": exp.name, "config": exp.__dict__, "total_M": total_m, "trainable_M": trainable_m,
              "size_MB_fp32": total_m * 4 * 1e6 / 2**20, "F1_at_0.5": metrics_default["F1"], "F1_final": metrics_final["F1"],
              "P": metrics_final["P"], "R": metrics_final["R"], "F1_per_class": metrics_final["F1_per_class"],
              "threshold": threshold, "train_min": history["train_sec"] / 60, "infer_ms_per_case": infer_sec_per_case * 1000,
              "peak_GB": peak_gb, "history": history, "section_probe": section_probe}
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"### {exp.name}: trainable {trainable_m:.2f}M/{total_m:.1f}M | dev macro F1 {metrics_default['F1']:.4f} @0.5, "
          f"{metrics_final['F1']:.4f} @thr {threshold:.2f} | P {metrics_final['P']:.3f} R {metrics_final['R']:.3f}")

    model.cpu(); del model; gc.collect(); torch.cuda.empty_cache()

In [29]:
lm = load_phase1_model(MAIN_MODEL).to(DEVICE).eval()
x = torch.tensor([[BOS] + train_split.token_ids[0][:CONTENT_LEN]], device=DEVICE)
with torch.no_grad():
    logits = lm(x[:, :-1])
lm_loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1)).item()
print(f"Phase-1 LM loss on a train case: {lm_loss:.3f} (random init ~ {math.log(VOCAB_SIZE):.1f})")
del lm; gc.collect(); torch.cuda.empty_cache()

Phase-1 LM loss on a train case: 3.182 (random init ~ 9.9)


In [52]:
run_experiment(make_exp("full_finetune", method="full", epochs=2, lr=3e-5, head_lr=5e-4,
                        batch_docs=4, grad_accum=8, export_teacher_logits=True))

cast 0 frozen Linear layers to fp16
[full_finetune] resumed after epoch 2/2 (best so far 0.6334 at epoch 2)


In [54]:
run_experiment(make_exp("kd_student_half_depth", method="full", block_stride=3, teacher_name="full_finetune",
                        epochs=1, lr=5e-5, head_lr=5e-4, batch_docs=4, grad_accum=8,
                        kd_temperature=2.0, kd_ce_weight=0.5))

cast 0 frozen Linear layers to fp16


[kd_student_half_depth] ep 1/1 loss 0.4322 | dev macro P 0.653 R 0.640 F1 0.6319


  -> ./cs781_phase4/submission_kd_student_half_depth.zip
### kd_student_half_depth: trainable 73.58M/73.6M | dev macro F1 0.6319 @0.5, 0.6580 @thr 0.38 | P 0.669 R 0.662


In [33]:
# run_experiment(make_exp("gradual_unfreeze", method="unfreeze", base_model= "mla-sin", epochs=1, lr=3e-5, head_lr=5e-4, batch_docs=4, grad_accum=8))
# run_experiment(make_exp("lora_r16", method="lora", base_model= "swiglu-moe-rope-conv",  lr=2e-4))
# run_experiment(make_exp("adalora_r16", method="adalora", batch_docs=4, base_model= "gqa-rope", epochs=1,  lr=2e-4, adalora_final_ratio=0.5))
# run_experiment(make_exp("adapter_64", method="adapter", base_model= "swiglu-moe-rope-conv", lr=5e-4, adapter_dim=64))
# run_experiment(make_exp(f"lorma_swiglu-moe-rope-conv", base_model="swiglu-moe-rope-conv", batch_docs=4, method="lorma", lr=2e-4))

# BitFit needs bias terms: gqa-rope has biased FFNs (the SwiGLU/MoE variants only have LayerNorm biases)
# run_experiment(make_exp("bitfit_gqa-rope", base_model="gqa-rope", method="bitfit", lr=1e-3, head_lr=1e-3))

cast 0 frozen Linear layers to fp16


[gradual_unfreeze] ep 1/1 loss 0.6757 | dev macro P 0.619 R 0.616 F1 0.6128


  -> ./cs781_phase4/submission_gradual_unfreeze.zip
### gradual_unfreeze: trainable 91.53M/91.5M | dev macro F1 0.6128 @0.5, 0.6149 @thr 0.47 | P 0.616 R 0.616
cast 48 frozen Linear layers to fp16


[adalora_r16] ep 1/1 loss 0.6865 | dev macro P 0.541 R 0.537 F1 0.5272 | active AdaLoRA rank 256/512


  -> ./cs781_phase4/submission_adalora_r16.zip
### adalora_r16: trainable 0.66M/66.4M | dev macro F1 0.5272 @0.5, 0.5541 @thr 0.53 | P 0.554 R 0.554


In [56]:
run_experiment(make_exp(f"lorma_swiglu-moe-rope-conv", base_model="swiglu-moe-rope-conv", epochs=1, batch_docs=4, method="lorma", lr=2e-4))

cast 216 frozen Linear layers to fp16


[lorma_swiglu-moe-rope-conv] ep 1/1 loss 0.6780 | dev macro P 0.574 R 0.573 F1 0.5710


  -> ./cs781_phase4/submission_lorma_swiglu-moe-rope-conv.zip
### lorma_swiglu-moe-rope-conv: trainable 0.79M/198.1M | dev macro F1 0.5710 @0.5, 0.5760 @thr 0.53 | P 0.581 R 0.579


In [31]:
# import gc
# import torch
# import IPython

# # 1. Clear notebook output history (wh1ich may retain tensors)
# try:
#     ip = get_ipython()
#     ip.run_line_magic("reset", "-f")
# except Exception:
#     pass

# # 2. Clear CUDA IPC references and Python garbage
# gc.collect()

# if torch.cuda.is_available():
#     try:
#         torch.cuda.synchronize()
#         torch.cuda.ipc_collect()
#     except Exception:
#         pass

# # 3. Delete remaining user-defined global variables
# # Keep only the modules needed for cleanup and reporting.
# _keep = {"gc", "torch", "IPython", "_keep"}

# for _name in list(globals().keys()):
#     if not _name.startswith("__") and _name not in _keep:
#         try:
#             del globals()[_name]
#         except Exception:
#             pass

NameError: name 'gc' is not defined

In [15]:
# run_experiment(make_exp("base_partial_top4", method="partial", epochs=5, lr=5e-5, head_lr=1e-3,
#                         n_top=4, run_section_probe=True))

[base_partial_top4] ep 1/5 loss 0.8452 | dev macro P 0.325 R 0.335 F1 0.3273


[base_partial_top4] ep 2/5 loss 0.6942 | dev macro P 0.329 R 0.361 F1 0.3333


[base_partial_top4] ep 3/5 loss 0.5045 | dev macro P 0.234 R 0.500 F1 0.3191


[base_partial_top4] ep 4/5 loss 0.5150 | dev macro P 0.234 R 0.500 F1 0.3191


[base_partial_top4] ep 5/5 loss 0.4634 | dev macro P 0.234 R 0.500 F1 0.3191


  -> ./cs781_phase4_smoke/submission_base_partial_top4.zip


### base_partial_top4: trainable 58.22M/190.0M | dev macro F1 0.3333 @0.5, 0.4375 @thr 0.45 | P 0.439 R 0.439


In [16]:
# for model_name in PHASE1_CONFIGS:
    # run_experiment(make_exp(f"lorma_{model_name}", base_model=model_name, method="lorma", lr=2e-4))

[lorma_gqa-rope] ep 1/5 loss 0.7661 | dev macro P 0.517 R 0.504 F1 0.3992


[lorma_gqa-rope] ep 2/5 loss 0.6912 | dev macro P 0.464 R 0.465 F1 0.4640


[lorma_gqa-rope] ep 3/5 loss 0.6401 | dev macro P 0.483 R 0.496 F1 0.3637


[lorma_gqa-rope] ep 4/5 loss 0.6180 | dev macro P 0.742 R 0.529 F1 0.3816


[lorma_gqa-rope] ep 5/5 loss 0.6200 | dev macro P 0.742 R 0.529 F1 0.3816


  -> ./cs781_phase4_smoke/submission_lorma_gqa-rope.zip
### lorma_gqa-rope: trainable 0.53M/66.3M | dev macro F1 0.4640 @0.5, 0.5625 @thr 0.49 | P 0.565 R 0.565


... (14 outputs left out)

[lorma_swiglu-moe-rope-conv] ep 3/5 loss 0.6275 | dev macro P 0.226 R 0.467 F1 0.3043


[lorma_swiglu-moe-rope-conv] ep 4/5 loss 0.6142 | dev macro P 0.234 R 0.500 F1 0.3191


[lorma_swiglu-moe-rope-conv] ep 5/5 loss 0.6191 | dev macro P 0.234 R 0.500 F1 0.3191


  -> ./cs781_phase4_smoke/submission_lorma_swiglu-moe-rope-conv.zip
### lorma_swiglu-moe-rope-conv: trainable 0.79M/198.1M | dev macro F1 0.3522 @0.5, 0.4684 @thr 0.58 | P 0.519 R 0.512


In [17]:
# Full fine-tuning: performance ceiling AND the KD teacher (exports its train-set logits)
run_experiment(make_exp("full_finetune", method="full", epochs=6, lr=3e-5, head_lr=5e-4,
                        batch_docs=2, grad_accum=8, export_teacher_logits=True))

run_experiment(make_exp("lora_r16", method="lora", lr=2e-4))
run_experiment(make_exp("adalora_r16", method="adalora", lr=2e-4, adalora_final_ratio=0.5))
run_experiment(make_exp("adapter_64", method="adapter", lr=5e-4, adapter_dim=64))
run_experiment(make_exp("gradual_unfreeze", method="unfreeze", epochs=6, lr=3e-5, head_lr=5e-4, batch_docs=2, grad_accum=8))

# BitFit needs bias terms: gqa-rope has biased FFNs (the SwiGLU/MoE variants only have LayerNorm biases)
run_experiment(make_exp("bitfit_gqa-rope", base_model="gqa-rope", method="bitfit", lr=1e-3, head_lr=1e-3))

[full_finetune] ep 1/6 loss 0.8779 | dev macro P 0.474 R 0.482 F1 0.4459


[full_finetune] ep 2/6 loss 0.6866 | dev macro P 0.609 R 0.537 F1 0.4589


[full_finetune] ep 3/6 loss 0.4915 | dev macro P 0.234 R 0.500 F1 0.3191


[full_finetune] ep 4/6 loss 0.4331 | dev macro P 0.742 R 0.529 F1 0.3816


[full_finetune] ep 5/6 loss 0.3536 | dev macro P 0.583 R 0.551 F1 0.4910


[full_finetune] ep 6/6 loss 0.3198 | dev macro P 0.479 R 0.484 F1 0.4421


... (29 outputs left out)

[bitfit_gqa-rope] ep 3/5 loss 0.6205 | dev macro P 0.541 R 0.522 F1 0.4459


[bitfit_gqa-rope] ep 4/5 loss 0.6154 | dev macro P 0.481 R 0.488 F1 0.4231


[bitfit_gqa-rope] ep 5/5 loss 0.6013 | dev macro P 0.479 R 0.484 F1 0.4421


  -> ./cs781_phase4_smoke/submission_bitfit_gqa-rope.zip
### bitfit_gqa-rope: trainable 0.05M/65.8M | dev macro F1 0.5077 @0.5, 0.5901 @thr 0.44 | P 0.608 R 0.602


In [18]:
# Teacher: full_finetune (12 blocks). Student: same Phase 1 model keeping every 2nd block (6 blocks), distilled from teacher logits.
run_experiment(make_exp("kd_student_half_depth", method="full", block_stride=2, teacher_name="full_finetune",
                        epochs=6, lr=5e-5, head_lr=5e-4, batch_docs=2, grad_accum=8,
                        kd_temperature=2.0, kd_ce_weight=0.5))

[kd_student_half_depth] ep 1/6 loss 0.9741 | dev macro P 0.266 R 0.500 F1 0.3469


[kd_student_half_depth] ep 2/6 loss 0.6022 | dev macro P 0.234 R 0.500 F1 0.3191


[kd_student_half_depth] ep 3/6 loss 0.4090 | dev macro P 0.226 R 0.467 F1 0.3043


[kd_student_half_depth] ep 4/6 loss 0.3118 | dev macro P 0.481 R 0.488 F1 0.4231


[kd_student_half_depth] ep 5/6 loss 0.2668 | dev macro P 0.234 R 0.500 F1 0.3191


[kd_student_half_depth] ep 6/6 loss 0.2406 | dev macro P 0.234 R 0.500 F1 0.3191


  -> ./cs781_phase4_smoke/submission_kd_student_half_depth.zip
### kd_student_half_depth: trainable 102.69M/102.7M | dev macro F1 0.4231 @0.5, 0.5393 @thr 0.65 | P 0.622 R 0.575


In [30]:
# Long-document representation (which part of the case is fed to the model)
for strategy in ["head", "tail", "uniform"]:
    run_experiment(make_exp(f"abl_strategy_{strategy}", strategy=strategy))

# Chunk pooling: attention-based and hierarchical (BiGRU over chunks)
for pooling in ["attn", "bigru"]:
    run_experiment(make_exp(f"abl_pooling_{pooling}", pooling=pooling))

# Class imbalance handling
for loss_type in ["ce", "focal"]:
    run_experiment(make_exp(f"abl_loss_{loss_type}", loss_type=loss_type))

cast 204 frozen Linear layers to fp16


[abl_strategy_head] ep 1/3 loss 0.7022 | dev macro P 0.230 R 0.500 F1 0.3151


[abl_strategy_head] ep 2/3 loss 0.6694 | dev macro P 0.656 R 0.562 F1 0.4644


[abl_strategy_head] ep 3/3 loss 0.6366 | dev macro P 0.562 R 0.555 F1 0.5308


  -> ./cs781_phase4_smoke/submission_abl_strategy_head.zip
### abl_strategy_head: trainable 0.79M/190.8M | dev macro F1 0.5308 @0.5, 0.5773 @thr 0.47 | P 0.595 R 0.590
cast 204 frozen Linear layers to fp16


[abl_strategy_tail] ep 1/3 loss 0.7010 | dev macro P 0.230 R 0.500 F1 0.3151


... (19 outputs left out)

[abl_loss_focal] ep 1/3 loss 0.1902 | dev macro P 0.557 R 0.556 F1 0.5496


[abl_loss_focal] ep 2/3 loss 0.1725 | dev macro P 0.506 R 0.504 F1 0.4423


[abl_loss_focal] ep 3/3 loss 0.1600 | dev macro P 0.533 R 0.533 F1 0.5300


  -> ./cs781_phase4_smoke/submission_abl_loss_focal.zip
### abl_loss_focal: trainable 0.79M/190.8M | dev macro F1 0.5496 @0.5, 0.5496 @thr 0.50 | P 0.557 R 0.556


In [20]:
def load_results():
    rows = []
    for file in sorted(os.listdir(OUT)):
        if file.startswith("result_") and file.endswith(".json"):
            r = json.load(open(f"{OUT}/{file}"))
            rows.append({"name": r["name"], "model": r["config"]["base_model"], "method": r["config"]["method"],
                         "pool": r["config"]["pooling"], "strategy": r["config"]["strategy"], "loss": r["config"]["loss_type"],
                         "F1@0.5": r["F1_at_0.5"], "F1": r["F1_final"], "P": r["P"], "R": r["R"],
                         "F1_c0": r["F1_per_class"][0], "F1_c1": r["F1_per_class"][1],
                         "total_M": r["total_M"], "trainable_M": r["trainable_M"], "size_MB": r["size_MB_fp32"],
                         "train_min": r["train_min"], "infer_ms/case": r["infer_ms_per_case"], "peak_GB": r["peak_GB"]})
    return pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)

results_df = load_results()
pd.set_option("display.width", 250, "display.max_columns", 30)
print(results_df.round(4).to_string())

                          name                 model    method   pool   strategy         loss  F1@0.5      F1       P       R   F1_c0   F1_c1   total_M  trainable_M   size_MB  train_min  infer_ms/case  peak_GB
0         abl_strategy_uniform   gqa-rope-swiglu-moe     lorma   mean    uniform  weighted_ce  0.5271  0.5934  0.5992  0.5980  0.5806  0.6061  190.8111       0.7880  727.8867     1.7314        86.4232   4.6325
1            abl_strategy_head   gqa-rope-swiglu-moe     lorma   mean       head  weighted_ce  0.5901  0.5901  0.6083  0.6020  0.5517  0.6286  190.8111       0.7880  727.8867     1.7273        86.6883   4.6325
2               abl_loss_focal   gqa-rope-swiglu-moe     lorma   mean  head_tail        focal  0.5625  0.5901  0.6083  0.6020  0.5517  0.6286  190.8111       0.7880  727.8867     1.7291        86.6371   4.6322
3            abl_strategy_tail   gqa-rope-swiglu-moe     lorma   mean       tail  weighted_ce  0.5901  0.5901  0.6083  0.6020  0.5517  0.6286  190.8111       0.

In [21]:
for file in sorted(os.listdir(OUT)):
    if file.startswith("result_"):
        r = json.load(open(f"{OUT}/{file}"))
        if r["section_probe"] is not None:
            print(r["name"], "- dev macro-F1 when only these chunks are shown at inference:")
            print(pd.Series(r["section_probe"]).round(4).to_string())

base_partial_top4 - dev macro-F1 when only these chunks are shown at inference:
head            0.3273
tail            0.3651
head_tail       0.3333
uniform         0.3522
first_third     0.3637
middle_third    0.3764
last_third      0.3098


## 10. Anaylsis and Submission

In [22]:
def error_analysis(name):
    probs = np.load(f"{OUT}/dev_probs_{name}.npy")
    threshold = json.load(open(f"{OUT}/result_{name}.json"))["threshold"]
    preds, labels = (probs >= threshold).astype(int), dev_split.labels
    lengths = np.array([len(t) for t in dev_split.token_ids])
    groups = {"TP": (preds == 1) & (labels == 1), "TN": (preds == 0) & (labels == 0),
              "FP": (preds == 1) & (labels == 0), "FN": (preds == 0) & (labels == 1)}
    rows = [{"group": g, "count": int(m.sum()), "mean_tokens": lengths[m].mean(), "mean_P(class1)": probs[m].mean()} for g, m in groups.items()]
    print(f"{name} (threshold {threshold:.2f})"); print(pd.DataFrame(rows).round(3).to_string(index=False))
    quartile = pd.qcut(lengths, 4, labels=["Q1 short", "Q2", "Q3", "Q4 long"])
    print(pd.Series(preds == labels).groupby(quartile, observed=True).mean().round(3).rename("accuracy by case length").to_string())

error_analysis(results_df.loc[0, "name"])      # best experiment on dev

abl_strategy_uniform (threshold 0.51)
group  count  mean_tokens  mean_P(class1)
   TP     10     4551.500           0.563
   TN      9     4882.111           0.444
   FP      8     3944.000           0.617
   FN      5     7896.400           0.450
Q1 short    0.625
Q2          0.625
Q3          0.625
Q4 long     0.500


In [23]:
def write_ensemble(names, ensemble_name="ensemble"):
    dev_probs  = np.mean([np.load(f"{OUT}/dev_probs_{n}.npy")  for n in names], axis=0)
    test_probs = np.mean([np.load(f"{OUT}/test_probs_{n}.npy") for n in names], axis=0)
    threshold, f1 = tune_threshold(dev_probs, dev_split.labels)
    print(f"{ensemble_name}: dev macro-F1 {f1:.4f} @thr {threshold:.2f}")
    write_submission((test_probs >= threshold).astype(int), ensemble_name)

write_ensemble(results_df["name"].head(4).tolist())

ensemble: dev macro-F1 0.5934 @thr 0.50
  -> ./cs781_phase4_smoke/submission_ensemble.zip
